In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys, numpy as np, pandas as pd, torch
sys.path.insert(0, "/content/drive/MyDrive/esci/code")
from metrics import evaluate
from cross_encoder import product_to_text, esci_row_to_product
import train_cross_encoder as T

BASE = "/content/drive/MyDrive/esci"
df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")

# ---- leak checks (test IDs are used only here, then dropped)
ids = {p: set(df.loc[df.part == p, "query_id"]) for p in ["train", "val", "test"]}
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"]), "query_id leak"
norm = lambda s: " ".join(str(s).lower().split())
same_text = set(df.loc[df.part == "train", "query"].map(norm)) & set(df.loc[df.part == "val", "query"].map(norm))
assert not same_text, f"{len(same_text)} val query texts also in train, e.g. {list(same_text)[:5]}"
print("leak checks passed:", {p: len(v) for p, v in ids.items()})

train = df[df.part == "train"].reset_index(drop=True)
val   = df[df.part == "val"].reset_index(drop=True)
del df                                                    # test rows are no longer in memory

for d in (train, val):
    prods = [esci_row_to_product(r) for r in d.to_dict("records")]
    for s in T.STYLES:
        d[f"text_{s}"] = [product_to_text(p, s) for p in prods]
print("train rows:", len(train), "| val rows:", len(val), "| GPU:", torch.cuda.get_device_name(0))
print("targets:", train.groupby("esci_label")["gain"].first().to_dict())
print(T.CFG)

Mounted at /content/drive
leak checks passed: {'train': 3714, 'val': 656, 'test': 1295}
train rows: 65852 | val rows: 11741 | GPU: Tesla T4
targets: {'C': 0.01, 'E': 1.0, 'I': 0.0, 'S': 0.1}
{'epochs': 3, 'lr': 2e-05, 'warmup': 0.1, 'weight_decay': 0.01, 'batch': 32, 'accum': 1, 'max_len': 256, 'eval_batch': 128, 'seed': 42}


In [2]:
T.train_run("cross-encoder/ms-marco-MiniLM-L-6-v2", "minilm_l6_ft", train, val, evaluate, BASE)

minilm_l6_ft: starting at epoch 1 from cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

epoch 1 done: loss 0.4679, 1.6 min
       style  ndcg@10    mrr    p@5
       title   0.9254 0.9420 0.8460
 title_brand   0.9248 0.9408 0.8466
        full   0.9252 0.9412 0.8460
full_bullets   0.9280 0.9407 0.8503
  epoch 2 step 500/2058 loss 0.3416
  epoch 2 step 1000/2058 loss 0.3372
  epoch 2 step 1500/2058 loss 0.3358
  epoch 2 step 2000/2058 loss 0.3328


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 2 done: loss 0.3335, 1.7 min
       style  ndcg@10    mrr    p@5
       title   0.9253 0.9415 0.8485
 title_brand   0.9261 0.9443 0.8473
        full   0.9264 0.9420 0.8476
full_bullets   0.9305 0.9480 0.8476
  epoch 3 step 500/2058 loss 0.3030
  epoch 3 step 1000/2058 loss 0.3046
  epoch 3 step 1500/2058 loss 0.3039
  epoch 3 step 2000/2058 loss 0.3017


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 3 done: loss 0.3015, 1.7 min
       style  ndcg@10    mrr    p@5
       title   0.9253 0.9415 0.8460
 title_brand   0.9271 0.9443 0.8470
        full   0.9268 0.9423 0.8482
full_bullets   0.9278 0.9441 0.8479
